## Import Library

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

---

## 与主表合并
###  1.处理sales_train

In [13]:
#检查有无缺少的日期
start_date = calendar['date'].min()
end_date = calendar['date'].max()
print(start_date, end_date)
expected_dates = pd.date_range(start=start_date, end=end_date)
missing_dates = expected_dates.difference(calendar['date'])
if missing_dates.empty:
    print("The calendar contains all dates within the given range.")
else:
    print("Missing dates in the calendar:")
    print(missing_dates)


2016-01-01 00:00:00 2024-12-31 00:00:00
The calendar contains all dates within the given range.


In [14]:
sales_train = pd.read_csv('sales_train.csv',parse_dates=['date'])
sales_test = pd.read_csv('sales_test.csv',parse_dates=['date'])
inventory = pd.read_csv('inventory.csv')

In [15]:
#合并 sales train, sales test与inventory
sales_train = pd.merge(sales_train, inventory, how='left', on =['unique_id','warehouse'])
sales_test = pd.merge(sales_test, inventory, how='left', on =['unique_id','warehouse'])
#sales_train
sales_train

,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,type_5_discount,type_6_discount,product_unique_id,name,L1_category_name_en,L2_category_name_en,L3_category_name_en,L4_category_name_en
0,4845,2024-03-10,Budapest_1,6436.0,16.34,646.26,1.00,0.00000,0.0,0.0,0.0,0.15312,0.0,0.0,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
1,4845,2021-05-25,Budapest_1,4663.0,12.63,455.96,1.00,0.00000,0.0,0.0,0.0,0.15025,0.0,0.0,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
2,4845,2021-12-20,Budapest_1,6507.0,34.55,455.96,1.00,0.00000,0.0,0.0,0.0,0.15025,0.0,0.0,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
3,4845,2023-04-29,Budapest_1,5463.0,34.52,646.26,0.96,0.20024,0.0,0.0,0.0,0.15312,0.0,0.0,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
4,4845,2022-04-01,Budapest_1,5997.0,35.92,486.41,1.00,0.00000,0.0,0.0,0.0,0.15649,0.0,0.0,2375,Croissant_35,Bakery,Bakery_L2_18,Bakery_L3_83,Bakery_L4_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4007414,4941,2023-06-21,Prague_1,9988.0,26.56,34.06,1.00,0.00000,0.0,0.0,0.0,0.00000,0.0,0.0,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1
4007415,4941,2023-06-24,Prague_1,8518.0,27.42,34.06,1.00,0.00000,0.0,0.0,0.0,0.00000,0.0,0.0,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1
4007416,4941,2023-06-23,Prague_1,10424.0,33.39,34.06,1.00,0.00000,0.0,0.0,0.0,0.00000,0.0,0.0,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1
4007417,4941,2023-06-22,Prague_1,10342.0,22.88,34.06,1.00,0.00000,0.0,0.0,0.0,0.00000,0.0,0.0,2422,Kohlrabi_9,Fruit and vegetable,Fruit and vegetable_L2_3,Fruit and vegetable_L3_114,Fruit and vegetable_L4_1


In [16]:
result = sales_train.reset_index().groupby(['warehouse']).agg(
    count = ('date','size'),
    first_date = ('date','min'),
    last_date = ('date','max'),    
    date_difference=('date', lambda x: x.max() - x.min()),
    var_sales = ('sell_price_main','var'),
    mean_sales = ('sales','mean'),
    mean_price = ('sell_price_main','mean'),
    skew_price = ('sell_price_main','skew'),    
    max_price = ('sell_price_main','max'),
    #kurtosis_sales = ('sell_price_main','kurtosis')
)
result

,count,first_date,last_date,date_difference,var_sales,mean_sales,mean_price,skew_price,max_price
warehouse,,,,,,,,,
Brno_1,643637,2020-08-01,2024-06-02,1401 days,5108.657471,165.814570,67.873869,2.826800,1015.51
Budapest_1,574582,2020-08-01,2024-06-02,1401 days,822068.394919,102.934222,918.184147,4.306405,21682.99
Frankfurt_1,198937,2021-12-08,2024-06-02,907 days,10.047742,46.116097,2.977863,6.456303,50.15
Munich_1,259333,2021-05-20,2024-06-02,1109 days,7.316594,99.519923,3.002412,4.059457,38.75
Prague_1,780566,2020-08-01,2024-06-02,1401 days,4496.440265,146.635053,65.941507,2.614916,1112.54
Prague_2,770709,2020-08-01,2024-06-02,1401 days,4287.697391,75.200678,65.624530,2.540218,994.18
Prague_3,779655,2020-08-01,2024-06-02,1401 days,4970.689358,78.314883,67.141527,2.560767,963.01


### missing sales

In [17]:
missing_values = sales_train.groupby('warehouse')['sales'].apply(lambda x: x.isna().sum())

print("Missing values by warehouse:")
print(missing_values)

missing_rows = sales_train[sales_train['sales'].isna()]
print("\nRows with missing sales values:")
print(missing_rows)

Missing values by warehouse:
warehouse
Brno_1          0
Budapest_1      0
Frankfurt_1     6
Munich_1       46
Prague_1        0
Prague_2        0
Prague_3        0
Name: sales, dtype: int64

Rows with missing sales values:
         unique_id       date    warehouse  total_orders  sales  \
154017         885 2021-05-23     Munich_1           NaN    NaN   
154099         885 2021-05-21     Munich_1           NaN    NaN   
154149         885 2021-05-24     Munich_1           NaN    NaN   
154155         885 2021-05-22     Munich_1           NaN    NaN   
429815        1237 2021-12-10  Frankfurt_1           NaN    NaN   
429860        1237 2021-12-09  Frankfurt_1           NaN    NaN   
733468         725 2021-06-27     Munich_1           NaN    NaN   
734198         725 2021-06-26     Munich_1           NaN    NaN   
738594        3778 2021-05-29     Munich_1           NaN    NaN   
738600        3778 2021-05-23     Munich_1           NaN    NaN   
738632        3778 2021-05-31     Munic

In [18]:
# Fill missing values by linear interpolation within each unique product group
sales_train_sorted = sales_train.groupby('unique_id', group_keys=False).apply(lambda x: x.sort_values('date'))

def fill_missing_values(group):
    group['sales'] = group['sales'].interpolate(method='linear')
    group['total_orders'] = group['total_orders'].interpolate(method='linear')
    return group

sales_train_filled = sales_train_sorted.groupby('unique_id').apply(fill_missing_values)


/var/folders/m9/_gpbs0bn61q7077ldd342jc00000gn/T/ipykernel_36427/1722360303.py:9: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  sales_train_filled = sales_train_sorted.groupby('unique_id').apply(fill_missing_values)


In [19]:
# check remain missing values
remaining_missing_values = sales_train_filled[sales_train_filled['sales'].isna() | sales_train_filled['total_orders'].isna()]
remaining_missing_values

,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,type_5_discount,type_6_discount,product_unique_id,name,L1_category_name_en,L2_category_name_en,L3_category_name_en,L4_category_name_en


In [20]:
#合并 sales train, sales test与calendar
sales_train_filled = pd.merge(sales_train_filled, calendar, how='left', on=['date', 'warehouse'])
#sales_test = pd.merge(sales_test, calendar, how='left', on=['date', 'warehouse'])
sales_train_filled

,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,...,all_saints'_day_holiday,ascension_day,corpus_christi,german_unity_day,reformation_day,epiphany,assumption_of_the_virgin_mary,peace_festival_in_augsburg,day_before_holiday,day_after_holiday
0,0,2022-07-18,Budapest_1,5289.0,3.97,710.89,0.09,0.0,0.0,0.00000,...,0,0,0,0,0,0,0,0,1.0,1.0
1,0,2022-07-19,Budapest_1,5255.0,73.36,710.89,1.00,0.0,0.0,0.00000,...,0,0,0,0,0,0,0,0,1.0,1.0
2,0,2022-07-20,Budapest_1,5334.0,558.09,710.89,0.96,0.0,0.0,0.45045,...,0,0,0,0,0,0,0,0,1.0,1.0
3,0,2022-07-21,Budapest_1,5459.0,14.03,710.89,0.06,0.0,0.0,0.45045,...,0,0,0,0,0,0,0,0,1.0,1.0
4,0,2022-07-22,Budapest_1,5461.0,558.53,710.89,0.97,0.0,0.0,0.45045,...,0,0,0,0,0,0,0,0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4007414,5431,2024-05-29,Prague_2,5772.0,6.50,135.71,1.00,0.0,0.0,0.00000,...,0,0,0,0,0,0,0,0,1.0,1.0
4007415,5431,2024-05-30,Prague_2,6566.0,13.60,135.71,1.00,0.0,0.0,0.00000,...,0,0,0,0,0,0,0,0,1.0,1.0
4007416,5431,2024-05-31,Prague_2,6985.0,3.36,135.71,1.00,0.0,0.0,0.00000,...,0,0,0,0,0,0,0,0,1.0,1.0
4007417,5431,2024-06-01,Prague_2,5929.0,11.63,135.71,1.00,0.0,0.0,0.00000,...,0,0,0,0,0,0,0,0,1.0,1.0


In [24]:
# One-hot encode the 'warehouse' column
encoded_warehouses = pd.get_dummies(sales_train_filled['warehouse'], prefix='warehouse')
sales_train_encoded = pd.concat([sales_train_filled.drop('warehouse', axis=1), encoded_warehouses], axis=1)
sales_train_encoded

,unique_id,date,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,type_3_discount,...,epiphany,assumption_of_the_virgin_mary,peace_festival_in_augsburg,warehouse_Brno_1,warehouse_Budapest_1,warehouse_Frankfurt_1,warehouse_Munich_1,warehouse_Prague_1,warehouse_Prague_2,warehouse_Prague_3
0,0,2022-07-18,5289.0,3.97,710.89,0.09,0.0,0.0,0.00000,0.0,...,0,0,0,0,1,0,0,0,0,0
1,0,2022-07-19,5255.0,73.36,710.89,1.00,0.0,0.0,0.00000,0.0,...,0,0,0,0,1,0,0,0,0,0
2,0,2022-07-20,5334.0,558.09,710.89,0.96,0.0,0.0,0.45045,0.0,...,0,0,0,0,1,0,0,0,0,0
3,0,2022-07-21,5459.0,14.03,710.89,0.06,0.0,0.0,0.45045,0.0,...,0,0,0,0,1,0,0,0,0,0
4,0,2022-07-22,5461.0,558.53,710.89,0.97,0.0,0.0,0.45045,0.0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4007414,5431,2024-05-29,5772.0,6.50,135.71,1.00,0.0,0.0,0.00000,0.0,...,0,0,0,0,0,0,0,0,1,0
4007415,5431,2024-05-30,6566.0,13.60,135.71,1.00,0.0,0.0,0.00000,0.0,...,0,0,0,0,0,0,0,0,1,0
4007416,5431,2024-05-31,6985.0,3.36,135.71,1.00,0.0,0.0,0.00000,0.0,...,0,0,0,0,0,0,0,0,1,0
4007417,5431,2024-06-01,5929.0,11.63,135.71,1.00,0.0,0.0,0.00000,0.0,...,0,0,0,0,0,0,0,0,1,0


In [25]:
sales_train_encoded.to_csv('sales_train_encoded.csv', index=True)

KeyboardInterrupt: 

In [26]:
sales_train_encoded.columns.tolist()

['unique_id',
 'date',
 'total_orders',
 'sales',
 'sell_price_main',
 'availability',
 'type_0_discount',
 'type_1_discount',
 'type_2_discount',
 'type_3_discount',
 'type_4_discount',
 'type_5_discount',
 'type_6_discount',
 'product_unique_id',
 'name',
 'L1_category_name_en',
 'L2_category_name_en',
 'L3_category_name_en',
 'L4_category_name_en',
 'holiday_name',
 'holiday',
 'shops_closed',
 'winter_school_holidays',
 'school_holidays',
 'year',
 'month',
 'day',
 'day_of_week',
 'new_years_day',
 'international_womens_day',
 'good_friday',
 'holy_saturday',
 'easter_day',
 'easter_monday',
 'labour_day',
 "mother's_day",
 'cyrila_a_metodej',
 'jan_hus',
 'den_ceske_statnosti',
 'den_vzniku_samostatneho_ceskoslovenskeho_statu',
 'den_boje_za_svobodu_a_demokracii',
 'christmas_eve',
 '1st_christmas_day',
 '2nd_christmas_day',
 'den_osvobozeni',
 'memorial_day_of_the_republic',
 'memorial_day_for_the_victims_of_the_communist_dictatorships',
 'memorial_day_for_the_victims_of_the_hol

### lag-93

In [27]:
solution = pd.read_csv('solution.csv')
solution

,id,sales_hat
0,1226_2024-06-03,0
1,1226_2024-06-11,0
2,1226_2024-06-13,0
3,1226_2024-06-15,0
4,1226_2024-06-09,0
...,...,...
47016,4572_2024-06-03,0
47017,3735_2024-06-04,0
47018,3735_2024-06-03,0
47019,2129_2024-06-03,0


In [27]:
calendar

,date,holiday_name,shops_closed,winter_school_holidays,school_holidays,warehouse,year,month,day,day_of_week,...,all_saints'_day_holiday,ascension_day,corpus_christi,german_unity_day,reformation_day,epiphany,assumption_of_the_virgin_mary,peace_festival_in_augsburg,day_before_holiday,day_after_holiday
0,2016-01-01,Not,1,0,0,Brno_1,2016,1,1,4,...,0,0,0,0,0,0,0,0,1.0,-1.0
1,2016-01-02,Not,0,0,0,Brno_1,2016,1,2,5,...,0,0,0,0,0,0,0,0,1.0,1.0
2,2016-01-03,Not,0,0,0,Brno_1,2016,1,3,6,...,0,0,0,0,0,0,0,0,1.0,1.0
3,2016-01-04,Not,0,0,0,Brno_1,2016,1,4,0,...,0,0,0,0,0,0,0,0,1.0,1.0
4,2016-01-05,Not,0,0,0,Brno_1,2016,1,5,1,...,0,0,0,0,0,0,0,0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23011,2024-12-27,Not,0,0,0,Prague_3,2024,12,27,4,...,0,0,0,0,0,0,0,0,1.0,1.0
23012,2024-12-28,Not,0,0,0,Prague_3,2024,12,28,5,...,0,0,0,0,0,0,0,0,1.0,1.0
23013,2024-12-29,Not,0,0,0,Prague_3,2024,12,29,6,...,0,0,0,0,0,0,0,0,1.0,1.0
23014,2024-12-30,Not,0,0,0,Prague_3,2024,12,30,0,...,0,0,0,0,0,0,0,0,1.0,1.0


In [28]:
sales_train_filled[sales_train_filled['date'] == '2023-01-01']

,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,...,christmas_holiday,1848_revolution_memorial_day_(extra_holiday),all_saints'_day_holiday,ascension_day,corpus_christi,german_unity_day,reformation_day,epiphany,assumption_of_the_virgin_mary,peace_festival_in_augsburg
979,5,2023-01-01,Frankfurt_1,985.0,45.41,2.23,1.00,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1522,6,2023-01-01,Munich_1,1216.0,23.04,1.71,1.00,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2519,7,2023-01-01,Munich_1,1216.0,5.12,1.14,1.00,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4610,9,2023-01-01,Prague_1,6696.0,74.27,36.61,1.00,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
5989,10,2023-01-01,Prague_3,3573.0,39.24,51.72,0.48,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4003574,5426,2023-01-01,Budapest_1,3973.0,42.73,2305.81,0.24,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4005198,5428,2023-01-01,Brno_1,6625.0,1.86,168.12,1.00,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4005776,5429,2023-01-01,Prague_1,6696.0,0.00,239.98,1.00,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4006344,5430,2023-01-01,Prague_3,3573.0,0.97,247.71,1.00,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# 02/08改动部分：
# Keep only the required columns to reduce memory usage
sales_train_filled = sales_train_filled[['unique_id', 'warehouse', 'date', 'sales']]

# Convert the date column to datetime if not already
sales_train_filled['date'] = pd.to_datetime(sales_train_filled['date'])

# Generate the full date range from min to max dates
date_range = pd.date_range(start=sales_train_filled['date'].min(), end=sales_train_filled['date'].max())

# Reduce memory by processing each warehouse separately
result_frames = []
for warehouse in sales_train_filled['warehouse'].unique():
    # Filter data for the current warehouse
    warehouse_data = sales_train_filled[sales_train_filled['warehouse'] == warehouse]
    
    # Generate unique combinations of unique_id and date for this warehouse
    full_combinations = pd.MultiIndex.from_product(
        [warehouse_data['unique_id'].unique(), date_range],
        names=['unique_id', 'date']
    ).to_frame(index=False)

    # Add the warehouse column to the full combinations
    full_combinations['warehouse'] = warehouse

    # Merge with the warehouse data
    merged_data = full_combinations.merge(warehouse_data, on=['unique_id', 'warehouse', 'date'], how='left')

    # Fill missing sales with 0
    merged_data['sales'].fillna(0, inplace=True)

    # Collect the result
    result_frames.append(merged_data)

# Concatenate all results
sales_complete = pd.concat(result_frames, ignore_index=True)

# Display the result
print(sales_complete)

# Save to a new CSV if needed
# sales_complete.to_csv('sales_data_filled_optimized.csv', index=False)



/var/folders/m9/_gpbs0bn61q7077ldd342jc00000gn/T/ipykernel_36427/1777282244.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales_train_filled['date'] = pd.to_datetime(sales_train_filled['date'])


         unique_id       date   warehouse  sales
0                0 2020-08-01  Budapest_1   0.00
1                0 2020-08-02  Budapest_1   0.00
2                0 2020-08-03  Budapest_1   0.00
3                0 2020-08-04  Budapest_1   0.00
4                0 2020-08-05  Budapest_1   0.00
...            ...        ...         ...    ...
7556775       5428 2024-05-29      Brno_1  11.68
7556776       5428 2024-05-30      Brno_1  10.29
7556777       5428 2024-05-31      Brno_1  16.19
7556778       5428 2024-06-01      Brno_1  14.52
7556779       5428 2024-06-02      Brno_1  12.05

[7556780 rows x 4 columns]


In [22]:
# Ensure the date column is in datetime format
sales_complete['date'] = pd.to_datetime(sales_complete['date'])

# Sort the data by unique_id, warehouse, and date to ensure correct lag calculation
sales_complete = sales_complete.sort_values(by=['unique_id', 'warehouse', 'date'])

# Create lag columns for each day from 1 to 93
for i in range(1, 94):
    sales_complete[f'sales_lag_{i}'] = sales_complete.groupby(['unique_id', 'warehouse'])['sales'].shift(i)

# Display the updated DataFrame
print(sales_complete.head())

# Save the result to a CSV if needed
# sales_complete.to_csv('sales_data_with_lags.csv', index=False)


   unique_id       date   warehouse  sales  sales_lag_1  sales_lag_2  \
0          0 2020-08-01  Budapest_1    0.0          NaN          NaN   
1          0 2020-08-02  Budapest_1    0.0          0.0          NaN   
2          0 2020-08-03  Budapest_1    0.0          0.0          0.0   
3          0 2020-08-04  Budapest_1    0.0          0.0          0.0   
4          0 2020-08-05  Budapest_1    0.0          0.0          0.0   

   sales_lag_3  sales_lag_4  sales_lag_5  sales_lag_6  ...  sales_lag_83  \
0          NaN          NaN          NaN          NaN  ...           NaN   
1          NaN          NaN          NaN          NaN  ...           NaN   
2          NaN          NaN          NaN          NaN  ...           NaN   
3          0.0          NaN          NaN          NaN  ...           NaN   
4          0.0          0.0          NaN          NaN  ...           NaN   

   sales_lag_84  sales_lag_85  sales_lag_86  sales_lag_87  sales_lag_88  \
0           NaN           NaN      

In [29]:
# Group by 'warehouse' and 'date', then calculate the sum of 'sales'
#warehouse_daily_sales = sales_train_filled.groupby(['warehouse', 'date'])['sales'].sum().reset_index()
#warehouse_daily_sales

# Display the result
#warehouse_daily_sales.head()

,warehouse,date,sales
0,Brno_1,2020-08-01,48027.55
1,Brno_1,2020-08-02,44969.77
2,Brno_1,2020-08-03,54473.14
3,Brno_1,2020-08-04,48647.92
4,Brno_1,2020-08-05,47563.47
...,...,...,...
8962,Prague_3,2024-05-29,44732.75
8963,Prague_3,2024-05-30,53670.85
8964,Prague_3,2024-05-31,62156.54
8965,Prague_3,2024-06-01,50633.23


In [30]:
#补充缺失的日期，标记sales为NA
def fill_and_print_missing_dates(group):
    # Generate a complete date range between the minimum and maximum dates
    full_date_range = pd.date_range(start=group['date'].min(), end=group['date'].max())

    # Find missing dates
    missing_dates = full_date_range.difference(group['date'])
    if not missing_dates.empty:
        print(f"Missing dates for warehouse {group['warehouse'].iloc[0]}:")
        print(missing_dates)

    # Reindex the group with the full date range to add missing dates
    group = group.set_index('date').reindex(full_date_range).reset_index()

    # Rename the reindexed column to 'date'
    group.rename(columns={'index': 'date'}, inplace=True)

    # Fill missing sales values with NaN to be handled later
    return group

# Apply the function to each warehouse
warehouse_daily_sales_filled = warehouse_daily_sales.groupby('warehouse', group_keys=False).apply(fill_and_print_missing_dates)
warehouse_daily_sales_filled



Missing dates for warehouse Frankfurt_1:
DatetimeIndex(['2021-12-11', '2021-12-12', '2021-12-13', '2021-12-14',
               '2021-12-15', '2021-12-16', '2021-12-17', '2021-12-18',
               '2021-12-19', '2021-12-20', '2021-12-21', '2021-12-22',
               '2021-12-23', '2021-12-24', '2021-12-25', '2021-12-26',
               '2021-12-27', '2021-12-28', '2021-12-29', '2021-12-30',
               '2021-12-31', '2022-01-01', '2022-01-02', '2022-01-03',
               '2022-01-04', '2022-01-05', '2022-01-06', '2022-01-07',
               '2022-01-08', '2022-01-09', '2022-01-10', '2022-01-11',
               '2022-01-12', '2022-01-13', '2022-01-14', '2022-01-15',
               '2022-01-16', '2022-01-17', '2022-01-18', '2022-01-19',
               '2022-01-20', '2022-01-21', '2022-01-22', '2022-01-23',
               '2022-01-24', '2022-01-25', '2022-01-26', '2022-01-27',
               '2022-01-28', '2022-01-29', '2022-01-30', '2022-01-31',
               '2023-05-18'],
      

,date,warehouse,sales
0,2020-08-01,Brno_1,48027.55
1,2020-08-02,Brno_1,44969.77
2,2020-08-03,Brno_1,54473.14
3,2020-08-04,Brno_1,48647.92
4,2020-08-05,Brno_1,47563.47
...,...,...,...
1397,2024-05-29,Prague_3,44732.75
1398,2024-05-30,Prague_3,53670.85
1399,2024-05-31,Prague_3,62156.54
1400,2024-06-01,Prague_3,50633.23


In [31]:
# Define the number of lag features to create
lag_range = 93

# Initialize a new DataFrame to store the lag features
warehouse_daily_sales_with_lags = warehouse_daily_sales_filled.copy()

# Add lag features for each warehouse
for lag in range(1, lag_range + 1):
    warehouse_daily_sales_with_lags[f'lag_{lag}'] = (
        warehouse_daily_sales_with_lags.groupby('warehouse')['sales'].shift(lag)
    )

# Display the first few rows with the new lag columns added
warehouse_daily_sales_with_lags


,date,warehouse,sales,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,...,lag_84,lag_85,lag_86,lag_87,lag_88,lag_89,lag_90,lag_91,lag_92,lag_93
0,2020-08-01,Brno_1,48027.55,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-08-02,Brno_1,44969.77,48027.55,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-08-03,Brno_1,54473.14,44969.77,48027.55,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-08-04,Brno_1,48647.92,54473.14,44969.77,48027.55,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-08-05,Brno_1,47563.47,48647.92,54473.14,44969.77,48027.55,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1397,2024-05-29,Prague_3,44732.75,46822.17,49669.45,49800.64,48918.73,65784.59,50913.85,47829.09,...,43715.54,42491.24,42963.26,48442.82,50962.68,61246.24,50153.22,44435.93,44644.72,46192.00
1398,2024-05-30,Prague_3,53670.85,44732.75,46822.17,49669.45,49800.64,48918.73,65784.59,50913.85,...,48259.66,43715.54,42491.24,42963.26,48442.82,50962.68,61246.24,50153.22,44435.93,44644.72
1399,2024-05-31,Prague_3,62156.54,53670.85,44732.75,46822.17,49669.45,49800.64,48918.73,65784.59,...,59251.38,48259.66,43715.54,42491.24,42963.26,48442.82,50962.68,61246.24,50153.22,44435.93
1400,2024-06-01,Prague_3,50633.23,62156.54,53670.85,44732.75,46822.17,49669.45,49800.64,48918.73,...,48166.66,59251.38,48259.66,43715.54,42491.24,42963.26,48442.82,50962.68,61246.24,50153.22


### 2 处理sales_test

In [ ]:
sales_test

,unique_id,date,warehouse,total_orders,sell_price_main,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,type_5_discount,type_6_discount,product_unique_id,name,L1_category_name_en,L2_category_name_en,L3_category_name_en,L4_category_name_en
0,1226,2024-06-03,Brno_1,8679.0,13.13,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,627,Croissant_9,Bakery,Bakery_L2_14,Bakery_L3_37,Bakery_L4_1
1,1226,2024-06-11,Brno_1,8795.0,13.13,0.15873,0.0,0.0,0.0,0.0,0.0,0.0,627,Croissant_9,Bakery,Bakery_L2_14,Bakery_L3_37,Bakery_L4_1
2,1226,2024-06-13,Brno_1,10009.0,13.13,0.15873,0.0,0.0,0.0,0.0,0.0,0.0,627,Croissant_9,Bakery,Bakery_L2_14,Bakery_L3_37,Bakery_L4_1
3,1226,2024-06-15,Brno_1,8482.0,13.13,0.15873,0.0,0.0,0.0,0.0,0.0,0.0,627,Croissant_9,Bakery,Bakery_L2_14,Bakery_L3_37,Bakery_L4_1
4,1226,2024-06-09,Brno_1,8195.0,13.13,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,627,Croissant_9,Bakery,Bakery_L2_14,Bakery_L3_37,Bakery_L4_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47016,4572,2024-06-03,Munich_1,5254.0,2.09,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,2245,Apple_123,Fruit and vegetable,Fruit and vegetable_L2_1,Fruit and vegetable_L3_31,Fruit and vegetable_L4_1
47017,3735,2024-06-04,Prague_1,9698.0,11.00,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,1833,Kiwi_18,Fruit and vegetable,Fruit and vegetable_L2_1,Fruit and vegetable_L3_39,Fruit and vegetable_L4_52
47018,3735,2024-06-03,Prague_1,10256.0,11.00,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,1833,Kiwi_18,Fruit and vegetable,Fruit and vegetable_L2_1,Fruit and vegetable_L3_39,Fruit and vegetable_L4_52
47019,2129,2024-06-03,Brno_1,8679.0,37.75,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,1074,Grape_15,Fruit and vegetable,Fruit and vegetable_L2_1,Fruit and vegetable_L3_12,Fruit and vegetable_L4_1


In [ ]:
sales_test= pd.merge(sales_test, calendar, how='left', on=['date', 'warehouse'])
sales_test

,unique_id,date,warehouse,total_orders,sell_price_main,type_0_discount,type_1_discount,type_2_discount,type_3_discount,type_4_discount,...,all_saints'_day_holiday,ascension_day,corpus_christi,german_unity_day,reformation_day,epiphany,assumption_of_the_virgin_mary,peace_festival_in_augsburg,day_before_holiday,day_after_holiday
0,1226,2024-06-03,Brno_1,8679.0,13.13,0.00000,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
1,1226,2024-06-11,Brno_1,8795.0,13.13,0.15873,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
2,1226,2024-06-13,Brno_1,10009.0,13.13,0.15873,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
3,1226,2024-06-15,Brno_1,8482.0,13.13,0.15873,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
4,1226,2024-06-09,Brno_1,8195.0,13.13,0.00000,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47016,4572,2024-06-03,Munich_1,5254.0,2.09,0.00000,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
47017,3735,2024-06-04,Prague_1,9698.0,11.00,0.00000,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
47018,3735,2024-06-03,Prague_1,10256.0,11.00,0.00000,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
47019,2129,2024-06-03,Brno_1,8679.0,37.75,0.00000,0.0,0.0,0.0,0.0,...,0,0,0,0,0,0,0,0,1.0,1.0
